# NB_process_erasure — GDPR Crypto-Shredding Pipeline

6-step erasure pipeline for a single GDPR erasure request:

| Step | Action |
|------|--------|
| 1 | Load + validate request (status='pending') |
| 2 | Immediate Gold suppression → INSERT into erasure_registry, UPDATE status='suppressed' |
| 3 | DEK deletion → shred_subject() + stamp shredded_at in subject_key_store |
| 4 | Physical DELETE from Bronze (JSON field scan) |
| 5 | Post-shred validation — verify decrypt returns '[ERASED]', write dq_result PASS/FAIL |
| 6 | Complete → UPDATE status='complete', log step |

## Parameters
- `REQUEST_ID` — UUID of the erasure request row in `monitoring.erasure_requests`
- `DRY_RUN` — `true` (default) prints full plan without writing anything
- `CATALOG` — Unity Catalog name (default `workspace`)
- `OPERATOR` — identifier of the person/job running the erasure (audit trail)

## Security Notes
- `subject_id` is coerced to `int()` at step 1 (dvdrental IDs are always integers).
  A non-integer subject_id raises `ValueError` — prevents SQL injection.
- All writes use `spark.createDataFrame()` for parameterized inserts, never f-string SQL with user data.
- DRY_RUN=true prints the full plan without any side effects.

In [ ]:
%run ./NB_catalog_helpers
%run ./NB_key_management_helpers

In [ ]:
# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
dbutils.widgets.text("REQUEST_ID", "",      "Erasure request UUID")
dbutils.widgets.text("CATALOG",    "workspace", "Unity Catalog name")
dbutils.widgets.dropdown("DRY_RUN", "true", ["true", "false"])
dbutils.widgets.text("OPERATOR",   "system",  "Operator identifier (audit)")

REQUEST_ID = dbutils.widgets.get("REQUEST_ID").strip()
CATALOG    = dbutils.widgets.get("CATALOG")
DRY_RUN    = dbutils.widgets.get("DRY_RUN").lower() == "true"
OPERATOR   = dbutils.widgets.get("OPERATOR").strip() or "system"

if not REQUEST_ID:
    raise ValueError("REQUEST_ID is required")

print(f"REQUEST_ID : {REQUEST_ID}")
print(f"CATALOG    : {CATALOG}")
print(f"DRY_RUN    : {DRY_RUN}")
print(f"OPERATOR   : {OPERATOR}")

In [ ]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
import uuid as _uuid
from datetime import datetime, timezone
from pyspark.sql import Row

ERASURE_REQUESTS_FQN  = f"{CATALOG}.monitoring.erasure_requests"
ERASURE_REGISTRY_FQN  = f"{CATALOG}.monitoring.erasure_registry"
ERASURE_AUDIT_FQN     = f"{CATALOG}.monitoring.erasure_audit_log"
ERASURE_RUN_ID        = str(_uuid.uuid4())


def log_step(step: str, evidence: str = None) -> None:
    """Append one row to erasure_audit_log (parameterized write — no SQL injection)."""
    row = Row(
        request_id   = REQUEST_ID,
        step         = step,
        completed_at = datetime.now(timezone.utc),
        evidence     = evidence,
        operator     = OPERATOR,
    )
    if DRY_RUN:
        print(f"  [DRY RUN] audit_log ← step={step!r} evidence={evidence!r}")
    else:
        spark.createDataFrame([row]).write.format("delta").mode("append").saveAsTable(ERASURE_AUDIT_FQN)
        print(f"  Logged step={step!r}")

In [ ]:
# ===========================================================================
# Step 1 — Load + validate request
# ===========================================================================
print("Step 1: Load + validate erasure request")

requests = spark.sql(
    f"SELECT * FROM {ERASURE_REQUESTS_FQN} WHERE request_id = '{REQUEST_ID}'"
).collect()

if not requests:
    raise ValueError(f"No erasure request found with request_id={REQUEST_ID!r}")

req = requests[0]

if req["status"] not in ("pending", "suppressed"):  # allow resume from suppressed
    raise ValueError(
        f"Request {REQUEST_ID} has status={req['status']!r}. "
        "Only 'pending' or 'suppressed' requests can be processed."
    )

# SECURITY: coerce subject_id to int — dvdrental IDs are always integers.
# Raises ValueError on non-integer, preventing SQL injection.
try:
    subject_id_int = int(req["subject_id"])
except (ValueError, TypeError) as exc:
    raise ValueError(
        f"subject_id={req['subject_id']!r} is not a valid integer. "
        "Erasure aborted to prevent injection."
    ) from exc

SUBJECT_ID   = str(subject_id_int)   # canonical string form
SUBJECT_TYPE = req["subject_type"]

print(f"  subject_id={SUBJECT_ID}, subject_type={SUBJECT_TYPE!r}, status={req['status']!r}")
log_step("step1_validate", f"subject_id={SUBJECT_ID}, subject_type={SUBJECT_TYPE}")

In [ ]:
# ===========================================================================
# Step 2 — Immediate Gold suppression
# ===========================================================================
print("Step 2: Gold suppression — insert into erasure_registry")

if DRY_RUN:
    print(f"  [DRY RUN] INSERT INTO {ERASURE_REGISTRY_FQN}: subject_id={SUBJECT_ID}, subject_type={SUBJECT_TYPE}")
    print(f"  [DRY RUN] UPDATE {ERASURE_REQUESTS_FQN} SET status='suppressed' WHERE request_id='{REQUEST_ID}'")
else:
    # Parameterized insert via createDataFrame
    registry_row = Row(
        subject_id    = SUBJECT_ID,
        subject_type  = SUBJECT_TYPE,
        suppressed_at = datetime.now(timezone.utc),
    )
    (
        spark.createDataFrame([registry_row])
        .write.format("delta").mode("append")
        .saveAsTable(ERASURE_REGISTRY_FQN)
    )

    # Mark request as suppressed — Gold is now blocking this subject
    spark.sql(f"""
        UPDATE {ERASURE_REQUESTS_FQN}
        SET status = 'suppressed'
        WHERE request_id = '{REQUEST_ID}'
    """)
    print(f"  Inserted into erasure_registry, request status → suppressed")

log_step("step2_gold_suppression", f"subject_id={SUBJECT_ID} added to erasure_registry")

In [ ]:
# ===========================================================================
# Step 3 — DEK deletion (crypto-shredding)
# ===========================================================================
print("Step 3: Crypto-shredding — delete DEK from secret scope")

if DRY_RUN:
    print(f"  [DRY RUN] shred_subject({SUBJECT_ID!r}, {SUBJECT_TYPE!r})")
else:
    shred_subject(SUBJECT_ID, SUBJECT_TYPE)
    print(f"  DEK deleted for subject_id={SUBJECT_ID}")

log_step("step3_dek_shred", f"DEK deleted: subject_id={SUBJECT_ID}, subject_type={SUBJECT_TYPE}")

In [ ]:
# ===========================================================================
# Step 4 — Physical DELETE from Bronze (JSON field scan)
# ===========================================================================
print("Step 4: Physical DELETE from Bronze tables")

# Determine which Bronze tables to scan based on subject_type.
# customer_id appears in after/before payloads for customer, rental, payment.
SUBJECT_FIELD_MAP = {
    "silver_customer": {
        "bronze_tables": ["customer", "rental", "payment"],
        "json_paths":    ["$.after.customer_id", "$.before.customer_id"],
    },
    "silver_staff": {
        "bronze_tables": ["staff", "rental", "payment"],
        "json_paths":    ["$.after.staff_id", "$.before.staff_id"],
    },
    "silver_address": {
        "bronze_tables": ["address", "customer", "staff", "store"],
        "json_paths":    ["$.after.address_id", "$.before.address_id"],
    },
}

scan_config = SUBJECT_FIELD_MAP.get(SUBJECT_TYPE)
if not scan_config:
    print(f"  No Bronze scan configured for subject_type={SUBJECT_TYPE!r} — skipping step 4")
    log_step("step4_bronze_delete", f"skipped — no scan config for {SUBJECT_TYPE}")
else:
    deleted_counts = {}
    for bronze_table in scan_config["bronze_tables"]:
        bronze_fqn = f"{CATALOG}.bronze.{bronze_table}"
        if not spark.catalog.tableExists(bronze_fqn):
            print(f"  {bronze_fqn} does not exist — skipping")
            continue

        # Build WHERE clause: match subject_id in any of the JSON paths
        # subject_id_int is validated integer — safe to embed
        json_conditions = " OR ".join(
            f"get_json_object(value, '{path}') = '{subject_id_int}'"
            for path in scan_config["json_paths"]
        )

        if DRY_RUN:
            count_row = spark.sql(f"""
                SELECT COUNT(*) AS n FROM {bronze_fqn}
                WHERE {json_conditions}
            """).collect()[0]
            print(f"  [DRY RUN] DELETE FROM {bronze_fqn} WHERE ... → would delete {count_row['n']} rows")
            deleted_counts[bronze_table] = count_row['n']
        else:
            spark.sql(f"DELETE FROM {bronze_fqn} WHERE {json_conditions}")
            print(f"  Deleted from {bronze_fqn}")
            deleted_counts[bronze_table] = -1  # count not available after DELETE without extra query

    log_step("step4_bronze_delete", f"tables={list(deleted_counts.keys())}")

In [ ]:
# ===========================================================================
# Step 5 — Post-shred validation
# ===========================================================================
print("Step 5: Post-shred validation")

# Attempt to retrieve DEK — should return None (shredded)
dek_map  = get_dek_map([subject_id_int], SUBJECT_TYPE)
dek_val  = dek_map.get(SUBJECT_ID)

# Synthesize a ciphertext and attempt decrypt
test_ct  = b"\x00" * 28   # 12-byte nonce + 16-byte tag minimum (will fail auth — doesn't matter)
result   = decrypt_value(test_ct, dek_val)

post_shred_pass = (result == "[ERASED]")
status   = "PASS" if post_shred_pass else "FAIL"

if DRY_RUN:
    print(f"  [DRY RUN] Would write dq_result: post_shred_unreadable={status}")
else:
    write_dq_result(
        spark, run_id=ERASURE_RUN_ID, layer="silver",
        table_name=SUBJECT_TYPE, check_name="post_shred_unreadable",
        status=status, observed_value=0.0 if post_shred_pass else 1.0,
        message=f"subject_id={SUBJECT_ID}: DEK {'absent (ERASED)' if post_shred_pass else 'STILL PRESENT — shred failed!'}",
    )

print(f"  post_shred_unreadable: {status}")
log_step("step5_validation", f"post_shred_unreadable={status}")

if not post_shred_pass:
    raise RuntimeError(
        f"Step 5 FAIL: DEK for subject_id={SUBJECT_ID} is still retrievable after shred! "
        "Halting erasure — investigate secret scope."
    )

In [ ]:
# ===========================================================================
# Step 6 — Complete
# ===========================================================================
print("Step 6: Mark request complete")

if DRY_RUN:
    print(f"  [DRY RUN] UPDATE {ERASURE_REQUESTS_FQN} SET status='complete' WHERE request_id='{REQUEST_ID}'")
else:
    spark.sql(f"""
        UPDATE {ERASURE_REQUESTS_FQN}
        SET status = 'complete'
        WHERE request_id = '{REQUEST_ID}'
    """)
    print(f"  Request {REQUEST_ID} → complete")

log_step("step6_complete", f"erasure_run_id={ERASURE_RUN_ID}")

print(f"\nErasure pipeline complete for request_id={REQUEST_ID}, subject_id={SUBJECT_ID}")
print(f"  DRY_RUN={DRY_RUN}  —  {'no changes written' if DRY_RUN else '6/6 steps completed'}")